# Week 6: Working with External Data & APIs

This notebook demonstrates how to fetch real-time data from external APIs and integrate it with your datasets.

## 1. Import Required Libraries

In [1]:
import requests
import pandas as pd
import json
import time
from datetime import datetime

/Users/anastasiiakulakova/repos/Practical-Data-Science-Course/new_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Understanding APIs

An API (Application Programming Interface) is a way for programs to communicate with each other and share data.

In [5]:
# Key API Concepts
api_concepts = {
    "Endpoint": "The URL where the API data is located",
    "Parameters": "Options that customize your request (country, year, format)",
    "JSON": "Data format used by most APIs (structured, human-readable)",
    "HTTP Status Codes": "Codes that tell you if the request was successful (200 = OK, 404 = Not Found)",
    "Rate Limit": "Maximum number of requests you can make per hour",
    "API Key": "Authentication token (some APIs require this, others don't)"
}

for concept, description in api_concepts.items():
    print(f"{concept}: {description}")

Endpoint: The URL where the API data is located
Parameters: Options that customize your request (country, year, format)
JSON: Data format used by most APIs (structured, human-readable)
HTTP Status Codes: Codes that tell you if the request was successful (200 = OK, 404 = Not Found)
Rate Limit: Maximum number of requests you can make per hour
API Key: Authentication token (some APIs require this, others don't)


## 3. Making Your First API Request

Let's start with a simple API that doesn't require authentication.

In [6]:
# Example: REST Countries API (no authentication required)
# This API provides information about countries

def fetch_country_data(country_name):
    """
    Fetch country information from REST Countries API
    
    Parameters:
    -----------
    country_name : str
        Name of the country (e.g., 'France')
    
    Returns:
    --------
    dict : Country data or None if request fails
    """
    try:
        # Make the API request
        url = f"https://restcountries.com/v3.1/name/{country_name}"
        response = requests.get(url)
        
        # Check if request was successful
        response.raise_for_status()
        
        # Parse JSON response
        data = response.json()
        
        return data[0] if data else None
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for {country_name}: {e}")
        return None

In [7]:
# Test the function
france_data = fetch_country_data('France')
france_data

{'tld': ['.fr'],
 'cca2': 'FR',
 'ccn3': '250',
 'cca3': 'FRA',
 'cioc': 'FRA',
 'independent': True,
 'status': 'officially-assigned',
 'unMember': True,
 'idd': {'root': '+3', 'suffixes': ['3']},
 'capital': ['Paris'],
 'altSpellings': ['FR', 'French Republic', 'République française'],
 'region': 'Europe',
 'subregion': 'Western Europe',
 'landlocked': False,
 'borders': ['AND', 'BEL', 'DEU', 'ITA', 'LUX', 'MCO', 'ESP', 'CHE'],
 'area': 543908.0,
 'maps': {'googleMaps': 'https://goo.gl/maps/g7QxxSFsWyTPKuzd7',
  'openStreetMaps': 'https://www.openstreetmap.org/relation/1403916'},
 'population': 66351959,
 'fifa': 'FRA',
 'car': {'signs': ['F'], 'side': 'right'},
 'timezones': ['UTC-10:00',
  'UTC-09:30',
  'UTC-09:00',
  'UTC-08:00',
  'UTC-04:00',
  'UTC-03:00',
  'UTC+01:00',
  'UTC+02:00',
  'UTC+03:00',
  'UTC+04:00',
  'UTC+05:00',
  'UTC+10:00',
  'UTC+11:00',
  'UTC+12:00'],
 'continents': ['Europe'],
 'flag': '🇫🇷',
 'name': {'common': 'France',
  'official': 'French Republic'

## 4. Working with JSON Data

Understanding JSON structure is key to working with APIs.

In [8]:
# Example JSON structure from an API response
example_json = """
{
  "name": "France",
  "capital": "Paris",
  "region": "Europe",
  "population": 67970571,
  "languages": {
    "fra": "French"
  },
  "borders": ["AND", "BEL", "DEU", "ITA", "LUX", "MCO", "ESP", "CHE"]
}
"""

print("Example JSON structure:")
print(example_json)

Example JSON structure:

{
  "name": "France",
  "capital": "Paris",
  "region": "Europe",
  "population": 67970571,
  "languages": {
    "fra": "French"
  },
  "borders": ["AND", "BEL", "DEU", "ITA", "LUX", "MCO", "ESP", "CHE"]
}



In [10]:
# Parse JSON string
data = json.loads(example_json)
print(f"\nAccessing data:")
print(f"Country: {data['name']}")
print(f"Capital: {data['capital']}")
print(f"Language: {data['languages']['fra']}")
# data['borders'])}")

type(data['borders'])


Accessing data:
Country: France
Capital: Paris
Language: French


list

## 5. Converting API Data to DataFrame

Transform JSON responses into tabular data for analysis.

In [15]:
url = (
    "https://restcountries.com/v3.1/all"
    "?fields=name,flags,car,borders,unMember,subregion"
)

response = requests.get(url)
response.raise_for_status()  # will raise if something goes wrong

countries = response.json()
countries_data = []

# Quick check for car driving side and borders
for country in countries:  # show first 10 as sample
    countries_data.append({
        "name": country.get("name", {}).get("common"),
        "flag_png": country.get("flags", {}).get("png"),
        "flag_svg": country.get("flags", {}).get("svg"),
        "car_signs": country.get("car", {}).get("signs"),
        "car_side": country.get("car", {}).get("side"),
        "borders": country.get("borders"),
        "un_member": country.get("unMember"),
        "region": country.get("region"),
        "subregion": country.get("subregion"),
    })

countries_df = pd.DataFrame(countries_data)
countries_df.head()


,name,flag_png,flag_svg,car_signs,car_side,borders,un_member,region,subregion
0,Zimbabwe,https://flagcdn.com/w320/zw.png,https://flagcdn.com/zw.svg,[ZW],left,"[BWA, MOZ, ZAF, ZMB]",True,None,Southern Africa
1,Kiribati,https://flagcdn.com/w320/ki.png,https://flagcdn.com/ki.svg,[KIR],left,[],True,None,Micronesia
2,Ghana,https://flagcdn.com/w320/gh.png,https://flagcdn.com/gh.svg,[GH],right,"[BFA, CIV, TGO]",True,None,Western Africa
3,North Korea,https://flagcdn.com/w320/kp.png,https://flagcdn.com/kp.svg,[],right,"[CHN, KOR, RUS]",True,None,Eastern Asia
4,Spain,https://flagcdn.com/w320/es.png,https://flagcdn.com/es.svg,[E],right,"[AND, FRA, GIB, PRT, MAR]",True,None,Southern Europe


In [5]:
len(countries_df)

250

In [16]:
# counts per subregion & car_side
counts = countries_df.groupby(['subregion', 'car_side']).size()

# percentages within each subregion
percentages = counts.groupby(level='subregion').apply(
    lambda x: 100 * x / x.sum()
)

percentages

# # nicer as a DataFrame
# percentages = percentages.reset_index(name='percent')
# percentages

subregion                  subregion                  car_side
                                                      right       100.000000
Australia and New Zealand  Australia and New Zealand  left        100.000000
Caribbean                  Caribbean                  left         53.571429
                                                      right        46.428571
Central America            Central America            right       100.000000
Central Asia               Central Asia               right       100.000000
Central Europe             Central Europe             right       100.000000
Eastern Africa             Eastern Africa             left         42.105263
                                                      right        57.894737
Eastern Asia               Eastern Asia               left         37.500000
                                                      right        62.500000
Eastern Europe             Eastern Europe             right       100.000000
Melanesia    

In [17]:
import pandas as pd
import plotly.express as px

# If you don't yet have a DataFrame from your list:
# countries_df = pd.DataFrame(countries_data)

# 1) Compute percentage of countries by car_side within each subregion
counts = (
    countries_df
    .groupby(['subregion', 'car_side'])
    .size()
    .reset_index(name='count')
)

percent_df = (
    counts
    .assign(
        percent=lambda d: 100 * d['count'] / d.groupby('subregion')['count'].transform('sum')
    )
)

fig = px.bar(
    percent_df,
    x='subregion',
    y='percent',
    color='car_side',
    barmode='stack',        # or 'group' if you prefer side-by-side bars
    title='Percentage of Countries by Driving Side per Subregion',
    labels={'percent': 'Percent of Countries', 'subregion': 'Subregion', 'car_side': 'Driving Side'}
)

fig.update_layout(xaxis_tickangle=45)
fig.show()

## 6. Error Handling with APIs

Always handle potential errors when working with external data sources.

In [33]:
import logging

logger = logging.getLogger(__name__)

def safe_api_request(url, max_retries=3):
    """
    Safely make API requests with error handling and retries
    
    Parameters:
    -----------
    url : str
        The API endpoint URL
    max_retries : int
        Maximum number of retry attempts
    
    Returns:
    --------
    dict or None : JSON response or None if request fails
    """
    for attempt in range(max_retries):
        logger.error("HTTP error on attempt %d", attempt)
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
            
        except requests.exceptions.Timeout:
            print(f"Request timed out. Attempt {attempt + 1} of {max_retries}")
            
        except requests.exceptions.ConnectionError:
            print(f"Connection error. Is the API server down? Attempt {attempt + 1} of {max_retries}")
            
        except requests.exceptions.HTTPError as e:
            if response.status_code == 404:
                print(f"Resource not found (404)")
                return None
            elif response.status_code == 429:
                print(f"Rate limit exceeded. Waiting before retry...")
                logger.warning("Rate limit exceeded. Waiting before retry...")
                time.sleep(2)
            else:
                print(f"HTTP Error: {e}")
                
        except json.JSONDecodeError:
            print(f"Could not parse JSON response")
            return None
            
    print(f"Failed after {max_retries} attempts")
    return None

In [34]:
country_name = 'InvalidCountryXYZ'
url = f'https://restcountries.com/v3.1/name/{country_name}'
result = safe_api_request(url)

HTTP error on attempt 0


Resource not found (404)


In [21]:
result

[{'tld': ['.fr'],
  'cca2': 'FR',
  'ccn3': '250',
  'cca3': 'FRA',
  'cioc': 'FRA',
  'independent': True,
  'status': 'officially-assigned',
  'unMember': True,
  'idd': {'root': '+3', 'suffixes': ['3']},
  'capital': ['Paris'],
  'altSpellings': ['FR', 'French Republic', 'République française'],
  'region': 'Europe',
  'subregion': 'Western Europe',
  'landlocked': False,
  'borders': ['AND', 'BEL', 'DEU', 'ITA', 'LUX', 'MCO', 'ESP', 'CHE'],
  'area': 543908.0,
  'maps': {'googleMaps': 'https://goo.gl/maps/g7QxxSFsWyTPKuzd7',
   'openStreetMaps': 'https://www.openstreetmap.org/relation/1403916'},
  'population': 66351959,
  'fifa': 'FRA',
  'car': {'signs': ['F'], 'side': 'right'},
  'timezones': ['UTC-10:00',
   'UTC-09:30',
   'UTC-09:00',
   'UTC-08:00',
   'UTC-04:00',
   'UTC-03:00',
   'UTC+01:00',
   'UTC+02:00',
   'UTC+03:00',
   'UTC+04:00',
   'UTC+05:00',
   'UTC+10:00',
   'UTC+11:00',
   'UTC+12:00'],
  'continents': ['Europe'],
  'flag': '🇫🇷',
  'name': {'common': 'Fr

## 7. Best Practices for API Integration

Follow these guidelines when working with APIs.

In [8]:
best_practices = {
    "DO": [
        "✓ Check API documentation before making requests",
        "✓ Cache responses locally to reduce API calls",
        "✓ Use try/except blocks for error handling",
        "✓ Respect rate limits - add delays between requests",
        "✓ Log requests for debugging",
        "✓ Test your code with small datasets first"
    ],
    "DON'T": [
        "✗ Hardcode API keys in your code",
        "✗ Make unnecessary API calls",
        "✗ Share API responses publicly (may contain sensitive data)",
        "✗ Ignore rate limits",
        "✗ Assume data will always be in the expected format",
        "✗ Make assumptions about data quality"
    ]
}

for category, items in best_practices.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  {item}")


DO:
  ✓ Check API documentation before making requests
  ✓ Cache responses locally to reduce API calls
  ✓ Use try/except blocks for error handling
  ✓ Respect rate limits - add delays between requests
  ✓ Log requests for debugging
  ✓ Test your code with small datasets first

DON'T:
  ✗ Hardcode API keys in your code
  ✗ Make unnecessary API calls
  ✗ Share API responses publicly (may contain sensitive data)
  ✗ Ignore rate limits
  ✗ Assume data will always be in the expected format
  ✗ Make assumptions about data quality


## 10. Assignment: Fetch and Merge API Data

**Task**: Create a notebook that fetches data from a public API and merges it with an existing dataset.

**Requirements**:
1. **Choose an API** from the recommended list (or find your own)
2. **Make API requests** to fetch relevant data
3. **Convert JSON to DataFrame** with proper column names
4. **Handle errors gracefully** with try/except blocks
5. **Merge with existing data** to create a richer dataset
6. **Analyze the merged data** and share insights
7. **Document your process** with clear explanations

**Tips**:
- Start with a simple API like REST Countries
- Test with a single country/record first
- Add error handling for missing data
- Respect API rate limits - don't hammer the server!
- Cache successful responses locally

In [ ]:
from decouple import Config, RepositoryEnv
# Note: You should have a .env file in the parent directory with the line:
# search_api_key=your_actual_api_key_here following the format in example.env
config = Config(RepositoryEnv("../../.env"))
API_KEY = config("search_api_key", default=None)

url = "https://www.searchapi.io/api/v1/search"
params = {
  "engine": "google_scholar",
  "q": "oil prices",
  "api_key": API_KEY  # Make sure to set this in your .env file
}

response = requests.get(url, params=params)
print(response.text)


{
  "search_metadata": {
    "id": "search_J70QVB41KphBJg321kqLy9mx",
    "status": "Success",
    "created_at": "2026-03-04T18:37:29Z",
    "request_time_taken": 2.17,
    "parsing_time_taken": 0.03,
    "total_time_taken": 2.2,
    "request_url": "https://scholar.google.com/scholar?q=oil+prices&hl=en",
    "html_url": "https://www.searchapi.io/api/v1/searches/search_J70QVB41KphBJg321kqLy9mx.html",
    "json_url": "https://www.searchapi.io/api/v1/searches/search_J70QVB41KphBJg321kqLy9mx"
  },
  "search_parameters": {
    "engine": "google_scholar",
    "q": "oil prices",
    "hl": "en"
  },
  "search_information": {
    "query_displayed": "oil prices",
    "total_results": 4550000,
    "page": 1,
    "time_taken_displayed": 0.06
  },
  "organic_results": [
    {
      "position": 1,
      "title": "Understanding crude oil prices",
      "data_cid": "WjcZ_2vgUCkJ",
      "link": "https://journals.sagepub.com/doi/abs/10.5547/ISSN0195-6574-EJ-Vol30-No2-9",
      "publication": "JD Hamilt

In [39]:
json_response = response.json()
json_response['organic_results'][0]

{'position': 1,
 'title': 'Understanding crude oil prices',
 'data_cid': 'WjcZ_2vgUCkJ',
 'link': 'https://journals.sagepub.com/doi/abs/10.5547/ISSN0195-6574-EJ-Vol30-No2-9',
 'publication': 'JD Hamilton - The energy journal, 2009 - journals.sagepub.com',
 'snippet': '… that the price … price-elasticity of short-run demand and supply, the vulnerability of supplies to disruptions, and the peak in US oil production account for the broad behavior of oil prices …',
 'inline_links': {'cited_by': {'cites_id': '2977126108137863002',
   'total': 1968,
   'link': 'https://scholar.google.com/scholar?cites=2977126108137863002&as_sdt=80005&sciodt=0,11&hl=en'},
  'versions': {'cluster_id': '2977126108137863002',
   'total': 31,
   'link': 'https://scholar.google.com/scholar?cluster=2977126108137863002&hl=en&as_sdt=0,11'},
  'related_articles_link': 'https://scholar.google.com/scholar?q=related:WjcZ_2vgUCkJ:scholar.google.com/&scioq=oil+prices&hl=en&as_sdt=0,11'},
 'resource': {'name': 'nber.org',
 

In [43]:
# Extract organic results from the Google Scholar API response
scholar_results = []

for result in json_response['organic_results']:
    scholar_results.append({
        'title': result.get('title'),
        'publication': result.get('publication'),
        'cited_by': result.get('inline_links', {}).get('cited_by', {}).get('total'),
        'link': result.get('link')
    })

scholar_df = pd.DataFrame(scholar_results)

# Extract year from publication_info string
scholar_df['year'] = scholar_df['publication'].str.extract(r'(\d{4})')

scholar_df

,title,publication,cited_by,link,year
0,Understanding crude oil prices,"JD Hamilton - The energy journal, 2009 - journ...",1968,https://journals.sagepub.com/doi/abs/10.5547/I...,2009
1,Oil prices and the stock prices of alternative...,"I Henriques, P Sadorsky - Energy Economics, 20...",1039,https://www.sciencedirect.com/science/article/...,2008
2,Oil price shocks: Causes and consequences,"L Kilian - Annu. Rev. Resour. Econ., 2014 - an...",540,https://www.annualreviews.org/content/journals...,2014
3,Oil prices and the global economy: Is it diffe...,"K Mohaddes, MH Pesaran - Energy Economics, 201...",218,https://www.sciencedirect.com/science/article/...,2017
4,Oil prices and the stock market,"RC Ready - Review of Finance, 2018 - academic....",483,https://academic.oup.com/rof/article-abstract/...,2018
5,Oil prices and stock markets: A review of the ...,"S Degiannakis, G Filis, V Arora - The Energy J...",366,https://journals.sagepub.com/doi/abs/10.5547/0...,2018
6,Nonlinearities and the macroeconomic effects o...,"JD Hamilton - Macroeconomic dynamics, 2011 - c...",678,https://www.cambridge.org/core/journals/macroe...,2011
7,"The great plunge in oil prices: Causes, conseq...","J Baffes, MA Kose, F Ohnsorge… - … , and Polic...",550,https://papers.ssrn.com/sol3/papers.cfm?abstra...,2015
8,Oil prices and the global economy,"MR Arezki, Z Jakab, MD Laxton, MA Matsumoto… -...",100,https://books.google.com/books?hl=en&lr=&id=Kh...,2017
9,Oil prices and real exchange rates,"SS Chen, HC Chen - Energy economics, 2007 - El...",951,https://www.sciencedirect.com/science/article/...,2007
